# Qwen3-4B · LoRA · SROIE receipts — Colab version

Trains a LoRA adapter on **Qwen/Qwen3-4B-Instruct-2507** to turn receipt OCR text into JSON
(`company`, `date`, `address`, `total`), then checks the whole pipeline:
data → training → adapter sanity check → evaluation of base vs. LoRA.
An optional last section serves the adapter with **vLLM** and runs the original `eval.py` / `bench.py`.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (an L4 or A100 runtime also works and is faster).

| Section | What happens | Rough time on a free T4 (estimate) |
|---|---|---|
| 1–3 | GPU check, write project files, install | 2–4 min |
| 4 | Download SROIE, build splits | < 1 min |
| 5–6 | QLoRA training, one adapter per rank in `RANKS` (476 receipts, 2 epochs) | ~25–30 min per rank, measured at ~26 s per training step on a T4 |
| 7–8 | Sanity check + Transformers evaluation (100 test receipts) | ~3–4 min per model |
| 10 | *Optional:* vLLM serving + `eval.py` + `bench.py` | ~10–20 min (vLLM install is large) |

By default this is the **full run** for ranks 8, 16 and 32 (about 1.5 h of training on a T4). Set `QUICK = True` in section 5
for a short pipeline check (128 receipts, 1 epoch, 20 test receipts).

The code cells in section 2 are the files of the `qwen-receipt-lora` project.

**Changed after the first experiments:** the prompt's `total` instruction used to say "digits only", which made the base model
drop the decimal point (`2080` for `20.80`). It now asks for the amount as printed. The prompt is also the training input, so
**adapters trained before this change must be retrained**, and results from before it aren't comparable.

## 1 · GPU check

In [1]:
import subprocess
q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,compute_cap",
                    "--format=csv,noheader,nounits"], capture_output=True, text=True)
assert q.returncode == 0, "No GPU found: Runtime -> Change runtime type -> T4 GPU, then rerun."
name, mem_mib, cap = [x.strip() for x in q.stdout.strip().splitlines()[0].split(",")]
GPU_MEM_GB = int(mem_mib) / 1024
BF16_NATIVE = float(cap) >= 8.0          # Ampere+ (L4, A100). T4 = 7.5 -> fp16
USE_QLORA = GPU_MEM_GB < 20              # 4-bit base model on 16 GB cards
print(f"{name}: {GPU_MEM_GB:.0f} GB, compute capability {cap}")
print(f"precision: {'bf16' if BF16_NATIVE else 'fp16'}   QLoRA: {USE_QLORA}")

Tesla T4: 15 GB, compute capability 7.5
precision: fp16   QLoRA: True


## 2 · Write the project files
Each cell below writes one file of the project. Run them all; you don't need to read them now.

In [2]:
%mkdir -p /content/qwen-receipt-lora/src /content/qwen-receipt-lora/scripts
%cd /content/qwen-receipt-lora

/content/qwen-receipt-lora


In [3]:
%%writefile requirements-train.txt
# training env (venv: .venv-train)
torch>=2.6
transformers>=5.0
trl>=1.0
peft>=0.17
datasets>=3.0
accelerate>=1.0
bitsandbytes>=0.45   # only needed for --qlora

Writing requirements-train.txt


In [4]:
%%writefile requirements-serve.txt
# serving + eval env (venv: .venv-serve). vLLM pins its own torch -> keep it separate.
vllm
openai>=1.40
rapidfuzz>=3.0

Writing requirements-serve.txt


In [5]:
%%writefile src/schema.py
"""Single source of truth for the task: fields, JSON schema and the prompt.

Training, evaluation and serving all import from here, so the model is always
evaluated with exactly the prompt it was trained on.
"""

import json

FIELDS = ["company", "date", "address", "total"]

# JSON schema used for (a) validating model output and (b) vLLM structured outputs.
RECEIPT_SCHEMA = {
    "type": "object",
    "properties": {f: {"type": ["string", "null"]} for f in FIELDS},
    "required": FIELDS,
    "additionalProperties": False,
}

SYSTEM_PROMPT = (
    "You extract key information from the OCR text of a scanned receipt.\n"
    "Return ONLY a JSON object with exactly these keys: "
    '"company", "date", "address", "total".\n'
    "- company: the name of the shop or company that issued the receipt\n"
    "- date: the transaction date, copied as printed\n"
    "- address: the address of the issuing company\n"
    "- total: the final amount paid as printed, keeping the decimal point "
    "(e.g. 9.00 or 127.20), without a currency symbol\n"
    "Use null if a field is not present."
)


def build_prompt(ocr_text: str) -> list[dict]:
    """Chat messages for one receipt (everything except the answer)."""
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Receipt OCR text:\n\n{ocr_text}"},
    ]


def to_target(gold: dict) -> str:
    """Canonical JSON string the model is trained to produce (fixed key order)."""
    return json.dumps({f: gold.get(f) for f in FIELDS}, ensure_ascii=False)

Writing src/schema.py


In [6]:
%%writefile src/data_prep.py
"""SROIE -> chat-format JSONL for TRL SFT + evaluation.

Input  (from github.com/zzzDavid/ICDAR-2019-SROIE, folder data/):
    box/XXX.csv   one OCR line per row: x1,y1,x2,y2,x3,y3,x4,y4,text
    key/XXX.json  {"company", "date", "address", "total"}

Output (data/processed/):
    train.jsonl / val.jsonl / test.jsonl, one receipt per line:
    {"id", "prompt": [system, user], "completion": [assistant], "gold": {...}, "ocr_text"}
    stats.json  dataset statistics + "how often is the gold value in the OCR text"

Usage:
    python src/data_prep.py --sroie-dir data/raw/ICDAR-2019-SROIE/data --out-dir data/processed
"""

import argparse
import json
import random
import re
import statistics
from pathlib import Path

from schema import FIELDS, build_prompt, to_target


# --------------------------------------------------------------------------- #
# 1. Parse OCR boxes
# --------------------------------------------------------------------------- #
def read_boxes(path: Path) -> list[dict]:
    """Read one SROIE box file -> list of {x0, y0, x1, y1, text}."""
    raw = path.read_bytes()
    try:
        text = raw.decode("utf-8")
    except UnicodeDecodeError:
        text = raw.decode("latin-1")

    boxes = []
    for line in text.splitlines():
        parts = line.strip().split(",", 8)  # text itself may contain commas
        if len(parts) < 9 or not parts[8].strip():
            continue
        try:
            xs = [int(parts[i]) for i in (0, 2, 4, 6)]
            ys = [int(parts[i]) for i in (1, 3, 5, 7)]
        except ValueError:
            continue
        boxes.append({"x0": min(xs), "y0": min(ys), "x1": max(xs), "y1": max(ys),
                      "text": parts[8].strip()})
    return boxes


# --------------------------------------------------------------------------- #
# 2. Serialize layout -> text (the "layout serialization" step)
# --------------------------------------------------------------------------- #
def serialize(boxes: list[dict], mode: str = "rows") -> str:
    """Turn 2-D boxes into 1-D text for the LLM.

    plain: one box per line, in the order the OCR engine produced them
    rows:  group boxes whose vertical centres are close into one visual row,
           left-to-right, joined with ' | '. Keeps "TOTAL ... 9.00" on one line,
           which is exactly the signal the model needs for the total field.
    """
    if mode == "plain" or not boxes:
        return "\n".join(b["text"] for b in boxes)

    heights = [b["y1"] - b["y0"] for b in boxes]
    tol = 0.5 * statistics.median(heights)

    rows: list[list[dict]] = []
    for b in sorted(boxes, key=lambda b: (b["y0"] + b["y1"]) / 2):
        cy = (b["y0"] + b["y1"]) / 2
        if rows:
            row_cy = statistics.mean((r["y0"] + r["y1"]) / 2 for r in rows[-1])
            if abs(cy - row_cy) <= tol:
                rows[-1].append(b)
                continue
        rows.append([b])

    return "\n".join(" | ".join(b["text"] for b in sorted(r, key=lambda b: b["x0"]))
                     for r in rows)


# --------------------------------------------------------------------------- #
# 3. Gold labels
# --------------------------------------------------------------------------- #
def read_key(path: Path) -> dict:
    data = json.loads(path.read_text(encoding="utf-8"))
    return {f: (data.get(f) or None) for f in FIELDS}


def norm(s: str) -> str:
    return re.sub(r"\s+", " ", s).strip().upper()


# --------------------------------------------------------------------------- #
# 4. Build examples, split, write
# --------------------------------------------------------------------------- #
def build_example(rid: str, ocr_text: str, gold: dict) -> dict:
    return {
        "id": rid,
        "prompt": build_prompt(ocr_text),
        "completion": [{"role": "assistant", "content": to_target(gold)}],
        "gold": gold,
        "ocr_text": ocr_text,
    }


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--sroie-dir", required=True, help="folder containing box/ and key/")
    ap.add_argument("--out-dir", default="data/processed")
    ap.add_argument("--layout", choices=["rows", "plain"], default="rows")
    ap.add_argument("--n-val", type=int, default=50)
    ap.add_argument("--n-test", type=int, default=100)
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    src, out = Path(args.sroie_dir), Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)

    examples, skipped = [], 0
    for key_path in sorted((src / "key").glob("*.json")):
        box_path = src / "box" / f"{key_path.stem}.csv"
        if not box_path.exists():
            skipped += 1
            continue
        boxes = read_boxes(box_path)
        if not boxes:
            skipped += 1
            continue
        examples.append(build_example(key_path.stem, serialize(boxes, args.layout),
                                      read_key(key_path)))

    # Split by receipt id with a fixed seed -> reproducible, no leakage.
    random.Random(args.seed).shuffle(examples)
    test = examples[: args.n_test]
    val = examples[args.n_test : args.n_test + args.n_val]
    train = examples[args.n_test + args.n_val :]

    for name, split in [("train", train), ("val", val), ("test", test)]:
        with open(out / f"{name}.jsonl", "w", encoding="utf-8") as f:
            for ex in split:
                f.write(json.dumps(ex, ensure_ascii=False) + "\n")

    # How often does the gold value literally appear in the OCR text?
    # This is a soft ceiling for exact match: if the label is not in the input,
    # the model has to *correct* OCR noise, not just copy.
    in_text = {f: 0 for f in FIELDS}
    present = {f: 0 for f in FIELDS}
    for ex in examples:
        flat = norm(ex["ocr_text"].replace(" | ", " ").replace("\n", " "))
        for f in FIELDS:
            v = ex["gold"][f]
            if v:
                present[f] += 1
                in_text[f] += norm(v) in flat

    chars = [len(ex["ocr_text"]) for ex in examples]
    stats = {
        "n_receipts": len(examples), "skipped": skipped, "layout": args.layout,
        "split": {"train": len(train), "val": len(val), "test": len(test)},
        "ocr_chars": {"mean": round(statistics.mean(chars)), "max": max(chars)},
        "approx_prompt_tokens_max": max(chars) // 3 + 250,
        "gold_found_verbatim_in_ocr": {f: round(in_text[f] / max(present[f], 1), 3)
                                       for f in FIELDS},
    }
    (out / "stats.json").write_text(json.dumps(stats, indent=2))
    print(json.dumps(stats, indent=2))


if __name__ == "__main__":
    main()

Writing src/data_prep.py


In [7]:
%%writefile src/train_lora.py
"""LoRA / QLoRA supervised fine-tuning of Qwen on SROIE with TRL.

Usage:
    # bf16 LoRA (24 GB GPU: RTX 4090 / L4 / A10)
    python src/train_lora.py --rank 16 --out outputs/lora-r16

    # QLoRA (16 GB GPU, e.g. Colab T4): base model in 4-bit, adapter in 16-bit
    python src/train_lora.py --rank 16 --qlora --out outputs/lora-r16

Experiments worth running (one flag each):
    --rank 8 / 16 / 32                capacity of the adapter
    --targets attn | all              which linear layers get adapters
    --full-loss                       also train on the prompt tokens
    --epochs 1 / 3                    under- vs over-fitting on 476 receipts
"""

import argparse
import json

import torch
from datasets import load_dataset
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

TARGETS = {
    # attention projections only (the original LoRA paper setting)
    "attn": ["q_proj", "k_proj", "v_proj", "o_proj"],
    # every linear layer in the transformer blocks (attention + MLP)
    "all": "all-linear",
}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default="Qwen/Qwen3-4B-Instruct-2507")
    ap.add_argument("--train", default="data/processed/train.jsonl")
    ap.add_argument("--val", default="data/processed/val.jsonl")
    ap.add_argument("--out", default="outputs/lora-r16")
    ap.add_argument("--rank", type=int, default=16)
    ap.add_argument("--alpha", type=int, default=0, help="default: 2 * rank")
    ap.add_argument("--dropout", type=float, default=0.05)
    ap.add_argument("--targets", choices=TARGETS, default="all")
    ap.add_argument("--epochs", type=float, default=2)
    ap.add_argument("--lr", type=float, default=2e-4)
    ap.add_argument("--batch-size", type=int, default=4)
    ap.add_argument("--grad-accum", type=int, default=4)
    ap.add_argument("--max-length", type=int, default=2048)
    ap.add_argument("--qlora", action="store_true", help="load base model in 4-bit")
    ap.add_argument("--full-loss", action="store_true",
                    help="compute loss on prompt+completion instead of completion only")
    ap.add_argument("--seed", type=int, default=42)
    args = ap.parse_args()

    # Native bf16 needs compute capability >= 8 (Ampere+). torch.cuda.is_bf16_supported()
    # also returns True on a T4 because it counts software emulation, which is very slow.
    use_bf16 = torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    dtype = torch.bfloat16 if use_bf16 else torch.float16 if torch.cuda.is_available() \
        else torch.float32

    # ---------------- data: keep only what the trainer needs ---------------- #
    ds = load_dataset("json", data_files={"train": args.train, "val": args.val})
    ds = ds.select_columns(["prompt", "completion"])
    # Conversational prompt-completion format: TRL applies the chat template and
    # (by default) masks the prompt so loss is computed on the JSON answer only.

    # ---------------- model ---------------- #
    tokenizer = AutoTokenizer.from_pretrained(args.model)
    quant = None
    if args.qlora:
        quant = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",          # NormalFloat4: suits normally-distributed weights
            bnb_4bit_use_double_quant=True,     # also quantise the quantisation constants
            bnb_4bit_compute_dtype=dtype,       # matmuls run in 16-bit
        )
    model = AutoModelForCausalLM.from_pretrained(
        args.model, dtype=dtype, quantization_config=quant,
        device_map="auto" if torch.cuda.is_available() else None,
    )

    # ---------------- LoRA ---------------- #
    # W' = W + (alpha / r) * B @ A, with A: r x d_in, B: d_out x r, B initialised to 0
    # -> at step 0 the model is exactly the base model.
    peft_config = LoraConfig(
        r=args.rank,
        lora_alpha=args.alpha or 2 * args.rank,
        lora_dropout=args.dropout,
        target_modules=TARGETS[args.targets],
        bias="none",
        task_type="CAUSAL_LM",
    )

    # ---------------- training config ---------------- #
    cfg = SFTConfig(
        output_dir=args.out,
        num_train_epochs=args.epochs,
        per_device_train_batch_size=args.batch_size,
        per_device_eval_batch_size=args.batch_size,
        gradient_accumulation_steps=args.grad_accum,   # effective batch = 4 * 4 = 16
        learning_rate=args.lr,                         # LoRA wants ~10x a full-FT LR
        lr_scheduler_type="cosine",
        warmup_steps=0.05,                             # float in [0,1) = ratio of total steps
        max_length=args.max_length,
        completion_only_loss=not args.full_loss,
        gradient_checkpointing=True,                   # trade compute for activation memory
        bf16=use_bf16,
        fp16=torch.cuda.is_available() and not use_bf16,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        report_to="none",
        seed=args.seed,
    )

    trainer = SFTTrainer(
        model=model,
        args=cfg,
        train_dataset=ds["train"],
        eval_dataset=ds["val"],
        processing_class=tokenizer,
        peft_config=peft_config,
    )
    # QLoRA + fp16 (T4): TRL casts the trainable LoRA params of a 4-bit model to bf16, but the
    # fp16 GradScaler cannot unscale bf16 gradients. Keep them in fp32 (the optimizer is only
    # created inside trainer.train(), so it sees the fp32 params).
    if args.qlora and not use_bf16:
        for p in trainer.model.parameters():
            if p.requires_grad:
                p.data = p.data.float()

    trainer.model.print_trainable_parameters()

    trainer.train()
    trainer.save_model(args.out)          # writes adapter_config.json + adapter_model.safetensors
    tokenizer.save_pretrained(args.out)

    with open(f"{args.out}/train_args.json", "w") as f:
        json.dump(vars(args), f, indent=2)
    print(f"adapter saved to {args.out}")


if __name__ == "__main__":
    main()

Writing src/train_lora.py


In [8]:
%%writefile src/infer_check.py
"""Sanity-check an adapter with plain Transformers + PEFT before serving it.

Answers "did training actually change the model?" in under a minute.
With --compare it also runs the base model with the adapter disabled.

Usage:
    python src/infer_check.py --adapter outputs/lora-r16 --n 3 --compare
"""

import argparse
import json

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def generate(model, tok, messages, max_new_tokens=200):
    ids = tok.apply_chat_template(messages, add_generation_prompt=True,
                                  return_tensors="pt", return_dict=True).to(model.device)
    with torch.no_grad():
        out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, ids["input_ids"].shape[1]:], skip_special_tokens=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--adapter", required=True)
    ap.add_argument("--base", default=None, help="default: read from adapter_config.json")
    ap.add_argument("--data", default="data/processed/val.jsonl")
    ap.add_argument("--n", type=int, default=3)
    ap.add_argument("--compare", action="store_true", help="also show base-model output")
    args = ap.parse_args()

    base = args.base or json.load(open(f"{args.adapter}/adapter_config.json"))[
        "base_model_name_or_path"]
    tok = AutoTokenizer.from_pretrained(base)
    dtype = ((torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
             if torch.cuda.is_available() else torch.float32)   # fp16 on T4 (no native bf16)
    model = AutoModelForCausalLM.from_pretrained(
        base, dtype=dtype, device_map="auto" if torch.cuda.is_available() else None)
    model = PeftModel.from_pretrained(model, args.adapter)
    model.eval()

    rows = [json.loads(l) for l in open(args.data, encoding="utf-8")][: args.n]
    for r in rows:
        print(f"\n=== receipt {r['id']} ===")
        print("gold :", json.dumps(r["gold"], ensure_ascii=False))
        print("lora :", generate(model, tok, r["prompt"]))
        if args.compare:
            with model.disable_adapter():          # same weights, adapter switched off
                print("base :", generate(model, tok, r["prompt"]))


if __name__ == "__main__":
    main()

Writing src/infer_check.py


In [9]:
%%writefile src/metrics.py
"""Metrics for structured extraction.

Three levels, from "is it usable at all" to "is every field right":
  1. json_valid      the output parses as JSON
  2. schema_valid    it has exactly the 4 keys, all string-or-null
  3. per-field       exact match (after light normalisation) and fuzzy similarity
  4. address_loose   address compared on letters and digits only, because SROIE address
                     labels are inconsistent about punctuation (trailing ".", "," spacing);
                     all_loose = all four fields right with the address scored this way
"""

import json
import re

from rapidfuzz.distance import Levenshtein

from schema import FIELDS


def parse_output(text: str) -> tuple[dict | None, bool]:
    """Return (parsed_dict_or_None, schema_ok). Tolerates ```json fences."""
    text = text.strip()
    fence = re.search(r"```(?:json)?\s*(.*?)```", text, re.S)
    if fence:
        text = fence.group(1).strip()
    try:
        obj = json.loads(text)
    except json.JSONDecodeError:
        # Fall back to the first {...} block (base models often add chatter).
        m = re.search(r"\{.*\}", text, re.S)
        if not m:
            return None, False
        try:
            obj = json.loads(m.group(0))
        except json.JSONDecodeError:
            return None, False
    if not isinstance(obj, dict):
        return None, False
    schema_ok = set(obj) == set(FIELDS) and all(
        v is None or isinstance(v, str) for v in obj.values())
    return obj, schema_ok


def normalize(field: str, value) -> str:
    if value is None:
        return ""
    s = re.sub(r"\s+", " ", str(value)).strip().upper()
    if field == "total":
        # "RM 9.00", "9.00", "9" -> "9.00"
        num = re.sub(r"[^\d.]", "", s.replace(",", ""))
        try:
            return f"{float(num):.2f}"
        except ValueError:
            return s
    return s


def similarity(a: str, b: str) -> float:
    """1 - normalised Levenshtein distance (ANLS-style, no threshold)."""
    if not a and not b:
        return 1.0
    return 1.0 - Levenshtein.normalized_distance(a, b)


def score_example(pred: dict | None, gold: dict) -> dict:
    out = {}
    for f in FIELDS:
        p = normalize(f, (pred or {}).get(f))
        g = normalize(f, gold.get(f))
        out[f"{f}_exact"] = float(p == g)
        out[f"{f}_sim"] = similarity(p, g)
    out["all_exact"] = float(all(out[f"{f}_exact"] for f in FIELDS))

    alnum = lambda s: re.sub(r"[^A-Z0-9]", "", s)
    pa = normalize("address", (pred or {}).get("address"))
    ga = normalize("address", gold.get("address"))
    out["address_loose"] = float(alnum(pa) == alnum(ga))
    out["all_loose"] = float(out["address_loose"] == 1.0 and
                             all(out[f"{f}_exact"] for f in FIELDS if f != "address"))
    return out


def aggregate(rows: list[dict]) -> dict:
    n = len(rows)
    keys = [k for k in rows[0] if isinstance(rows[0][k], (int, float))]
    agg = {k: round(sum(r[k] for r in rows) / n, 4) for k in keys}
    agg["macro_exact"] = round(sum(agg[f"{f}_exact"] for f in FIELDS) / len(FIELDS), 4)
    agg["macro_sim"] = round(sum(agg[f"{f}_sim"] for f in FIELDS) / len(FIELDS), 4)
    agg["n"] = n
    return agg

Writing src/metrics.py


In [10]:
%%writefile src/hf_eval.py
"""Evaluate the base model and a LoRA adapter with plain Transformers (no vLLM needed).

Added for the Colab notebook. Uses the same prompts (from the JSONL), the same metrics
(metrics.py) and writes the same result files as eval.py, so report.py compares them.
The base model is evaluated by switching the adapter off on the same loaded weights.

Usage:
    python src/hf_eval.py --adapter outputs/lora-r16 --n 50
"""

import argparse
import json
import time
from pathlib import Path

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

from metrics import aggregate, parse_output, score_example
from schema import FIELDS


def is_strict_json(text: str) -> bool:
    try:
        return isinstance(json.loads(text), dict)
    except json.JSONDecodeError:
        return False


def generate_batch(model, tok, batch, max_new_tokens):
    texts = [tok.apply_chat_template(ex["prompt"], add_generation_prompt=True, tokenize=False)
             for ex in batch]
    enc = tok(texts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id)
    n_in = enc["input_ids"].shape[1]   # left padding -> every answer starts at n_in
    return [tok.decode(o[n_in:], skip_special_tokens=True).strip() for o in out]


def evaluate(model, tok, examples, tag, model_name, args):
    rows = []
    t0 = time.perf_counter()
    for i in range(0, len(examples), args.batch_size):
        batch = examples[i : i + args.batch_size]
        tb = time.perf_counter()
        outs = generate_batch(model, tok, batch, args.max_new_tokens)
        per_req = (time.perf_counter() - tb) / len(batch)
        for ex, text in zip(batch, outs):
            pred, schema_ok = parse_output(text)
            row = {"id": ex["id"], "raw": text, "pred": pred, "gold": ex["gold"],
                   "strict_json": float(is_strict_json(text)),
                   "json_valid": float(pred is not None),
                   "schema_valid": float(schema_ok),
                   "latency_s": per_req}
            row.update(score_example(pred, ex["gold"]))
            rows.append(row)
        print(f"  {tag}: {len(rows)}/{len(examples)}", flush=True)
    wall = time.perf_counter() - t0

    metrics = aggregate([{k: v for k, v in r.items()
                          if isinstance(v, float) and k != "latency_s"} for r in rows])
    metrics.update({"tag": tag, "model": model_name, "structured": False, "engine": "transformers",
                    "wall_s": round(wall, 2), "req_per_s": round(len(rows) / wall, 2)})

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    with open(out / f"{tag}.predictions.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    (out / f"{tag}.metrics.json").write_text(json.dumps(metrics, indent=2))

    print(f"\n== {tag} ==")
    print(f"strict_json {metrics['strict_json']:.3f}   json_valid {metrics['json_valid']:.3f}"
          f"   schema_valid {metrics['schema_valid']:.3f}")
    for f in FIELDS:
        print(f"{f:<8} exact {metrics[f + '_exact']:.3f}   sim {metrics[f + '_sim']:.3f}")
    print(f"address  ignoring punctuation {metrics['address_loose']:.3f}"
          f"   all fields (address ignoring punctuation) {metrics['all_loose']:.3f}")
    print(f"macro    exact {metrics['macro_exact']:.3f}   sim {metrics['macro_sim']:.3f}"
          f"   all-fields-correct {metrics['all_exact']:.3f}\n")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--adapter", required=True)
    ap.add_argument("--base", default=None, help="default: read from adapter_config.json")
    ap.add_argument("--data", default="data/processed/test.jsonl")
    ap.add_argument("--n", type=int, default=50)
    ap.add_argument("--batch-size", type=int, default=8)
    ap.add_argument("--max-new-tokens", type=int, default=256)
    ap.add_argument("--skip-base", action="store_true")
    ap.add_argument("--tag", default=None, help="default: <adapter folder>-hf")
    ap.add_argument("--out-dir", default="results")
    args = ap.parse_args()

    base = args.base or json.load(open(f"{args.adapter}/adapter_config.json"))[
        "base_model_name_or_path"]
    tok = AutoTokenizer.from_pretrained(base)
    tok.padding_side = "left"             # decoder-only batching: pad on the left
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    dtype = ((torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
             if torch.cuda.is_available() else torch.float32)
    model = AutoModelForCausalLM.from_pretrained(
        base, dtype=dtype, device_map="auto" if torch.cuda.is_available() else None)
    model = PeftModel.from_pretrained(model, args.adapter)
    model.eval()

    examples = [json.loads(l) for l in open(args.data, encoding="utf-8")][: args.n]
    tag = args.tag or f"{Path(args.adapter).name}-hf"

    if not args.skip_base:
        with model.disable_adapter():
            evaluate(model, tok, examples, "base-hf", base, args)
    evaluate(model, tok, examples, tag, args.adapter, args)


if __name__ == "__main__":
    main()

Writing src/hf_eval.py


In [11]:
%%writefile src/eval.py
"""Evaluate any model served behind an OpenAI-compatible API (vLLM) on SROIE.

The same script evaluates the base model and every LoRA adapter: with vLLM an
adapter is just another value of the `model` field in the request.

Usage:
    python src/eval.py --model Qwen/Qwen3-4B-Instruct-2507 --tag base
    python src/eval.py --model Qwen/Qwen3-4B-Instruct-2507 --tag base-json --structured
    python src/eval.py --model sroie-r16 --tag lora-r16
    python src/eval.py --model sroie-r16 --tag lora-r16-json --structured
"""

import argparse
import asyncio
import json
import time
from pathlib import Path

from openai import AsyncOpenAI

from metrics import aggregate, parse_output, score_example
from schema import FIELDS, RECEIPT_SCHEMA


def _is_strict_json(text: str) -> bool:
    try:
        return isinstance(json.loads(text), dict)
    except json.JSONDecodeError:
        return False


async def predict_one(client, sem, model, ex, structured, max_tokens):
    kwargs = {}
    if structured:
        # vLLM compiles the JSON schema into a grammar and masks invalid tokens
        # at every decoding step -> output is guaranteed to match the schema.
        kwargs["response_format"] = {
            "type": "json_schema",
            "json_schema": {"name": "receipt", "schema": RECEIPT_SCHEMA},
        }
    async with sem:
        t0 = time.perf_counter()
        resp = await client.chat.completions.create(
            model=model,
            messages=ex["prompt"],
            temperature=0.0,
            max_tokens=max_tokens,
            **kwargs,
        )
        latency = time.perf_counter() - t0
    text = resp.choices[0].message.content or ""
    pred, schema_ok = parse_output(text)
    row = {
        "id": ex["id"],
        "raw": text,
        "pred": pred,
        "gold": ex["gold"],
        "strict_json": float(_is_strict_json(text)),   # parses with no cleanup at all
        "json_valid": float(pred is not None),         # parses after stripping fences/chatter
        "schema_valid": float(schema_ok),
        "latency_s": latency,
        "completion_tokens": resp.usage.completion_tokens if resp.usage else 0,
    }
    row.update(score_example(pred, ex["gold"]))
    return row


async def run(args):
    examples = [json.loads(l) for l in open(args.data, encoding="utf-8")]
    if args.max_samples:
        examples = examples[: args.max_samples]

    client = AsyncOpenAI(base_url=args.base_url, api_key="EMPTY")
    sem = asyncio.Semaphore(args.concurrency)

    t0 = time.perf_counter()
    rows = await asyncio.gather(*[
        predict_one(client, sem, args.model, ex, args.structured, args.max_tokens)
        for ex in examples
    ])
    wall = time.perf_counter() - t0

    metrics = aggregate([{k: v for k, v in r.items()
                          if isinstance(v, float) and k != "latency_s"} for r in rows])
    lat = sorted(r["latency_s"] for r in rows)
    metrics.update({
        "tag": args.tag, "model": args.model, "structured": args.structured,
        "wall_s": round(wall, 2),
        "req_per_s": round(len(rows) / wall, 2),
        "latency_p50_s": round(lat[len(lat) // 2], 3),
        "latency_p95_s": round(lat[int(len(lat) * 0.95) - 1], 3),
    })

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    with open(out / f"{args.tag}.predictions.jsonl", "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    (out / f"{args.tag}.metrics.json").write_text(json.dumps(metrics, indent=2))

    print(f"\n== {args.tag}  ({args.model}, structured={args.structured}) ==")
    print(f"strict_json {metrics['strict_json']:.3f}   json_valid {metrics['json_valid']:.3f}"
          f"   schema_valid {metrics['schema_valid']:.3f}")
    for f in FIELDS:
        print(f"{f:<8} exact {metrics[f + '_exact']:.3f}   sim {metrics[f + '_sim']:.3f}")
    print(f"address  ignoring punctuation {metrics['address_loose']:.3f}"
          f"   all fields (address ignoring punctuation) {metrics['all_loose']:.3f}")
    print(f"macro    exact {metrics['macro_exact']:.3f}   sim {metrics['macro_sim']:.3f}"
          f"   all-fields-correct {metrics['all_exact']:.3f}")
    print(f"{metrics['req_per_s']} req/s, p50 {metrics['latency_p50_s']}s, "
          f"p95 {metrics['latency_p95_s']}s")


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", required=True, help="base model id or LoRA adapter name")
    ap.add_argument("--tag", required=True, help="name for this run's result files")
    ap.add_argument("--data", default="data/processed/test.jsonl")
    ap.add_argument("--base-url", default="http://localhost:8000/v1")
    ap.add_argument("--structured", action="store_true", help="use JSON-schema decoding")
    ap.add_argument("--concurrency", type=int, default=16)
    ap.add_argument("--max-tokens", type=int, default=256)
    ap.add_argument("--max-samples", type=int, default=0)
    ap.add_argument("--out-dir", default="results")
    asyncio.run(run(ap.parse_args()))


if __name__ == "__main__":
    main()

Writing src/eval.py


In [12]:
%%writefile src/report.py
"""Collect results/*.metrics.json into one comparison table + show errors.

Usage:
    python src/report.py                     # comparison table (markdown)
    python src/report.py --errors lora-r16   # worst predictions of one run
"""

import argparse
import json
from pathlib import Path

from schema import FIELDS

COLS = ["strict_json", "json_valid", "schema_valid"] + [f"{f}_exact" for f in FIELDS] + \
       ["address_loose", "macro_exact", "macro_sim", "all_exact", "all_loose",
        "req_per_s", "latency_p50_s"]


def table(results_dir: Path):
    runs = [json.loads(p.read_text()) for p in sorted(results_dir.glob("*.metrics.json"))]
    if not runs:
        print("no results yet")
        return
    print("| run | " + " | ".join(COLS) + " |")
    print("|---|" + "---|" * len(COLS))
    for r in runs:
        print(f"| {r['tag']} | " + " | ".join(str(r.get(c, "")) for c in COLS) + " |")


def errors(results_dir: Path, tag: str, k: int):
    rows = [json.loads(l) for l in open(results_dir / f"{tag}.predictions.jsonl")]
    rows.sort(key=lambda r: sum(r[f"{f}_sim"] for f in FIELDS))
    for r in rows[:k]:
        print(f"\n--- {r['id']}  json_valid={r['json_valid']}")
        for f in FIELDS:
            if not r[f"{f}_exact"]:
                p = (r["pred"] or {}).get(f)
                print(f"  {f:<8} pred={p!r}\n  {'':<8} gold={r['gold'][f]!r}")
        if r["pred"] is None:
            print("  raw:", r["raw"][:300])


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--results-dir", default="results")
    ap.add_argument("--errors", help="run tag to inspect")
    ap.add_argument("-k", type=int, default=10)
    a = ap.parse_args()
    if a.errors:
        errors(Path(a.results_dir), a.errors, a.k)
    else:
        table(Path(a.results_dir))

Writing src/report.py


In [13]:
%%writefile src/bench.py
"""Throughput / latency benchmark against a running vLLM server.

Measures, per (model-mix, concurrency) cell:
  TTFT  time to first token    ~ prefill cost (reading the ~700-token receipt)
  TPOT  time per output token  ~ decode cost
  req/s and output tok/s       ~ what continuous batching buys you

Usage:
    python src/bench.py --models Qwen/Qwen3-4B-Instruct-2507             # base only
    python src/bench.py --models sroie-r16                                # one adapter
    python src/bench.py --models sroie-r8,sroie-r32                       # 2 adapters mixed
    python src/bench.py --models Qwen/Qwen3-4B-Instruct-2507,sroie-r16 --concurrency 1,8,32
"""

import argparse
import asyncio
import json
import statistics
import time
from pathlib import Path

from openai import AsyncOpenAI


async def one_request(client, model, messages, max_tokens):
    t0 = time.perf_counter()
    ttft, n_tokens = None, 0
    stream = await client.chat.completions.create(
        model=model, messages=messages, temperature=0.0, max_tokens=max_tokens,
        stream=True, stream_options={"include_usage": True},
    )
    async for chunk in stream:
        if chunk.choices and chunk.choices[0].delta.content and ttft is None:
            ttft = time.perf_counter() - t0
        if chunk.usage:
            n_tokens = chunk.usage.completion_tokens
    total = time.perf_counter() - t0
    ttft = ttft or total
    tpot = (total - ttft) / max(n_tokens - 1, 1)
    return {"ttft": ttft, "total": total, "tokens": n_tokens, "tpot": tpot}


async def run_cell(client, models, prompts, concurrency, max_tokens):
    sem = asyncio.Semaphore(concurrency)

    async def worker(i, msgs):
        async with sem:
            # round-robin across models -> mixed-adapter batches on the server
            return await one_request(client, models[i % len(models)], msgs, max_tokens)

    t0 = time.perf_counter()
    res = await asyncio.gather(*[worker(i, p) for i, p in enumerate(prompts)])
    wall = time.perf_counter() - t0
    p50 = lambda xs: statistics.median(xs)
    return {
        "models": "+".join(models), "concurrency": concurrency, "n": len(res),
        "req_per_s": round(len(res) / wall, 2),
        "out_tok_per_s": round(sum(r["tokens"] for r in res) / wall, 1),
        "ttft_p50_ms": round(1000 * p50([r["ttft"] for r in res]), 1),
        "tpot_p50_ms": round(1000 * p50([r["tpot"] for r in res]), 2),
        "latency_p50_s": round(p50([r["total"] for r in res]), 3),
    }


async def main_async(args):
    rows = [json.loads(l) for l in open(args.data, encoding="utf-8")]
    prompts = [r["prompt"] for r in rows][: args.n]
    client = AsyncOpenAI(base_url=args.base_url, api_key="EMPTY")
    models = args.models.split(",")

    # warm-up: loads adapters into GPU slots, compiles kernels, fills caches
    await asyncio.gather(*[one_request(client, m, prompts[0], 8) for m in models])

    results = []
    for c in [int(x) for x in args.concurrency.split(",")]:
        cell = await run_cell(client, models, prompts, c, args.max_tokens)
        results.append(cell)
        print(json.dumps(cell))

    out = Path(args.out_dir)
    out.mkdir(parents=True, exist_ok=True)
    (out / f"bench_{'+'.join(m.split('/')[-1] for m in models)}.json").write_text(
        json.dumps(results, indent=2))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--models", required=True, help="comma-separated model / adapter names")
    ap.add_argument("--concurrency", default="1,4,16,32")
    ap.add_argument("--n", type=int, default=64, help="requests per cell")
    ap.add_argument("--max-tokens", type=int, default=128)
    ap.add_argument("--data", default="data/processed/test.jsonl")
    ap.add_argument("--base-url", default="http://localhost:8000/v1")
    ap.add_argument("--out-dir", default="results")
    asyncio.run(main_async(ap.parse_args()))


if __name__ == "__main__":
    main()

Writing src/bench.py


In [14]:
%%writefile src/merge_lora.py
"""Merge a LoRA adapter into the base weights: W_merged = W + (alpha/r) * B @ A.

Merged  -> one normal model, zero adapter overhead, but one full copy per task.
Adapter -> one base model shared by many tasks, small per-request overhead.
Serve both with vLLM and compare with bench.py to see the trade-off.

Usage:
    python src/merge_lora.py --adapter outputs/lora-r16 --out outputs/merged-r16
"""

import argparse
import json

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--adapter", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--base", default=None)
    args = ap.parse_args()

    base = args.base or json.load(open(f"{args.adapter}/adapter_config.json"))[
        "base_model_name_or_path"]
    # Merge in 16-bit on the original (non-quantised) weights, even after QLoRA.
    dtype = ((torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16)
             if torch.cuda.is_available() else torch.float32)   # fp16 on T4 (no native bf16)
    model = AutoModelForCausalLM.from_pretrained(base, dtype=dtype)
    model = PeftModel.from_pretrained(model, args.adapter)
    merged = model.merge_and_unload()
    merged.save_pretrained(args.out)
    AutoTokenizer.from_pretrained(base).save_pretrained(args.out)
    print(f"merged model saved to {args.out}")


if __name__ == "__main__":
    main()

Writing src/merge_lora.py


In [15]:
%%writefile scripts/get_data.sh
#!/usr/bin/env bash
# Download SROIE (626 labelled receipts, cleaned version) and build the JSONL splits.
set -euo pipefail
mkdir -p data/raw
if [[ ! -d data/raw/ICDAR-2019-SROIE ]]; then
  git clone --depth 1 https://github.com/zzzDavid/ICDAR-2019-SROIE.git data/raw/ICDAR-2019-SROIE
fi
python src/data_prep.py --sroie-dir data/raw/ICDAR-2019-SROIE/data --out-dir data/processed

Writing scripts/get_data.sh


In [16]:
%%writefile scripts/serve.sh
#!/usr/bin/env bash
# Serve the base model plus LoRA adapters from ONE vLLM server.
# Each adapter is addressable by name in the OpenAI "model" field.
#
#   bash scripts/serve.sh                      # base + adapters found in outputs/
#   ADAPTERS="sroie-r16=outputs/lora-r16" bash scripts/serve.sh
set -euo pipefail

BASE=${BASE:-Qwen/Qwen3-4B-Instruct-2507}
ADAPTERS=${ADAPTERS:-"sroie-r8=outputs/lora-r8 sroie-r16=outputs/lora-r16 sroie-r32=outputs/lora-r32"}

# keep only adapters that exist on disk
MODULES=()
for kv in $ADAPTERS; do
  [[ -d "${kv#*=}" ]] && MODULES+=("$kv") || echo "skip ${kv%%=*} (not trained yet)"
done

LORA_ARGS=()
if [[ ${#MODULES[@]} -gt 0 ]]; then
  LORA_ARGS=(--enable-lora
             --lora-modules "${MODULES[@]}"
             --max-lora-rank 32          # must be >= the largest adapter rank you serve
             --max-loras 4               # adapters resident on GPU at the same time (per batch)
             --max-cpu-loras 8)          # adapters cached in CPU RAM for fast swap-in
fi

exec vllm serve "$BASE" \
  --max-model-len 4096 \
  --gpu-memory-utilization 0.85 \
  --port 8000 \
  "${LORA_ARGS[@]}" \
  ${EXTRA_ARGS:-}
# On a T4 (no bf16) add:  EXTRA_ARGS="--dtype half"

Writing scripts/serve.sh


In [17]:
%%writefile scripts/run_experiments.sh
#!/usr/bin/env bash
# Full evaluation matrix. Start the server first:  bash scripts/serve.sh
set -euo pipefail
BASE=${BASE:-Qwen/Qwen3-4B-Instruct-2507}

# 1) base model: free text vs. JSON-schema constrained decoding
python src/eval.py --model "$BASE" --tag base
python src/eval.py --model "$BASE" --tag base-json --structured

# 2) every adapter the server knows about, with and without constrained decoding
for name in $(curl -s localhost:8000/v1/models | python -c \
    "import sys,json;print(' '.join(m['id'] for m in json.load(sys.stdin)['data'] if m.get('parent')))"); do
  python src/eval.py --model "$name" --tag "$name"
  python src/eval.py --model "$name" --tag "$name-json" --structured
done

# 3) serving performance: base alone, one adapter, two adapters mixed in the same batches
python src/bench.py --models "$BASE"
python src/bench.py --models sroie-r16
python src/bench.py --models sroie-r8,sroie-r32

python src/report.py

Writing scripts/run_experiments.sh


## 3 · Install the training stack
Colab already ships PyTorch; this adds TRL, PEFT, bitsandbytes and upgrades Transformers.
`rapidfuzz` is added here because `metrics.py` (used by `hf_eval.py`) needs it.
Everything runs as separate `python` processes, so no runtime restart is needed after the upgrade.

Colab also ships an old `torchao` (0.10). PEFT raises an error when it finds one older than 0.16, as soon as an adapter is loaded onto a
non-quantized model (sanity check, evaluation, merge). This project doesn't use torchao, so it's removed.

In [18]:
!pip install -q -r requirements-train.txt rapidfuzz
!pip uninstall -y -q torchao
!python -c "import torch, transformers, trl, peft; print('torch', torch.__version__, '| transformers', transformers.__version__, '| trl', trl.__version__, '| peft', peft.__version__)"

torch 2.11.0+cu130 | transformers 5.20.2 | trl 1.16.0 | peft 0.21.2


## 4 · Data pipeline

In [19]:
!bash scripts/get_data.sh
!head -n 128 data/processed/train.jsonl > data/processed/train_quick.jsonl
!wc -l data/processed/*.jsonl

Cloning into 'data/raw/ICDAR-2019-SROIE'...
remote: Enumerating objects: 1993, done.
remote: Counting objects: 100% (1993/1993), done.
remote: Compressing objects: 100% (1879/1879), done.
remote: Total 1993 (delta 112), reused 1971 (delta 110), pack-reused 0 (from 0)
Receiving objects: 100% (1993/1993), 278.24 MiB | 17.44 MiB/s, done.
Resolving deltas: 100% (112/112), done.
Updating files: 100% (1980/1980), done.
{
  "n_receipts": 626,
  "skipped": 0,
  "layout": "rows",
  "split": {
    "train": 476,
    "val": 50,
    "test": 100
  },
  "ocr_chars": {
    "mean": 720,
    "max": 1496
  },
  "approx_prompt_tokens_max": 748,
  "gold_found_verbatim_in_ocr": {
    "company": 0.974,
    "date": 0.994,
    "address": 0.776,
    "total": 0.998
  }
}
    100 data/processed/test.jsonl
    476 data/processed/train.jsonl
    128 data/processed/train_quick.jsonl
     50 data/processed/val.jsonl
    754 total


In [20]:
import json
ex = json.loads(open("data/processed/train.jsonl").readline())
print(ex["prompt"][1]["content"][:700], "...\n")
print("TARGET:", ex["completion"][0]["content"])

Receipt OCR text:

R
KEDAI PAPAN YEW CHUAN
(0005583085-K)
LOT 276 JALAN BANTING
43800 DENGKIL, SELANGOR.
TEL : 03-87686092 | FAX : 03-87686092
GST ID : 000781500416
TAX INVOICE
BILL TO :
SUCI ALAM JAYA TRANSPORT
NO 1 JALAN AMAN 2
TAMAN DESA 43800
DENGKIL SELANGOR
017-6057770
DOC NO. | : CS00010527 | DATE: 06/03/2018
CASHIER | : USER | TIME: 08:58:00
SALESPERSON : | REF. :
(GST) | (GST)
ITEM | QTY | S/PRICE | S/PRICE | AMOUNT | TAX
101869 | 2 | 60.00 | 63.60 | 127.20 | SR
PASIR KASAR (D)
TOTAL QTY: | 2 | 127.20
TOTAL SALES (EXCLUDING GST) : | 120.00
DISCOUNT : | 0.00
TOTAL GST : | 7.20
ROUNDING : | 0.00
TOTAL SALES (INCLUSIVE OF GST) : | 127.20
CASH : | 150.00
CHANGE : | 22.80
GST SUMMARY
TAX ...

TARGET: {"company": "KEDAI PAPAN YEW CHUAN", "date": "06/03/2018", "address": "LOT 276 JALAN BANTING 43800 DENGKIL, SELANGOR.", "total": "127.20"}


## 5 · Configure the run
`QUICK = False` is the full run: 476 training receipts, 2 epochs (about 60 optimiser steps), 100 test receipts.
`QUICK = True` is only a pipeline check: 128 receipts, 1 epoch (8 steps), 20 test receipts. That's too few to compare ranks.

`RANKS` lists the adapters to train, one after another. On a free T4 each rank takes about 25–30 minutes. If your session might
disconnect, train fewer ranks per session (for example `RANKS = [16]`) and save them to Drive (section 9) in between.

In [38]:
QUICK = False
RANKS = [8, 16, 32]                      # one adapter per rank -> outputs/lora-r8, outputs/lora-r16, ...
TRAIN_FILE = "data/processed/train_quick.jsonl" if QUICK else "data/processed/train.jsonl"
EPOCHS = 1 if QUICK else 2
EVAL_N = 20 if QUICK else 100            # test receipts to evaluate

def train_flags(rank):
    flags = ["--rank", str(rank), "--train", TRAIN_FILE, "--epochs", str(EPOCHS), "--out", f"outputs/lora-r{rank}"]
    if USE_QLORA:
        flags += ["--qlora", "--batch-size", "2", "--grad-accum", "8"]   # same effective batch of 16
    return " ".join(flags)

for r in RANKS:
    print("python src/train_lora.py", train_flags(r))

python src/train_lora.py --rank 8 --train data/processed/train.jsonl --epochs 2 --out outputs/lora-r8 --qlora --batch-size 2 --grad-accum 8
python src/train_lora.py --rank 16 --train data/processed/train.jsonl --epochs 2 --out outputs/lora-r16 --qlora --batch-size 2 --grad-accum 8
python src/train_lora.py --rank 32 --train data/processed/train.jsonl --epochs 2 --out outputs/lora-r32 --qlora --batch-size 2 --grad-accum 8


## 6 · Train the LoRA adapters
Trains every rank in `RANKS`. For each one, watch the `trainable params` line (r8: 16,515,072 · r16: 33,030,144 · r32: 66,060,288),
a falling `loss`, and `eval_loss` at the end of each epoch.

In [31]:
for r in RANKS:
    flags_r = train_flags(r)
    print(f"\n===== training rank {r} =====")
    !python src/train_lora.py {flags_r}


===== training rank 16 =====
Loading weights: 100% 398/398 [00:31<00:00, 12.66it/s]
trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
{'loss': '0.3867', 'grad_norm': '0.6228', 'learning_rate': '0.0001998', 'entropy': '0.004458', 'mean_token_accuracy': '0.975', 'num_tokens': '4.956e+04', 'epoch': '0.1681'}
{'loss': '0.06291', 'grad_norm': '0.2014', 'learning_rate': '0.0001946', 'entropy': '0.03019', 'mean_token_accuracy': '0.989', 'num_tokens': '9.686e+04', 'epoch': '0.3361'}
{'loss': '0.04072', 'grad_norm': '0.1031', 'learning_rate': '0.0001822', 'entropy': '0.07314', 'mean_token_accuracy': '0.9913', 'num_tokens': '1.47e+05', 'epoch': '0.5042'}
{'loss': '0.03268', 'grad_norm': '0.1576', 'learning_rate': '0.0001636', 'entropy': '0.03228', 'mean_token_accuracy': '0.993', 'num_tokens': '1.977e+05', 'epoch': '0.6723'}
{'loss': '0.04605', 'grad_norm': '0.1442', 'learning_rate': '0.0001402', 'entropy': '0.0248', 'mean_token_accuracy': '0.9923', 'num_tokens': '2

In [32]:
!ls -la outputs/lora-r*/adapter_model.safetensors

-rw------- 1 root root 132187888 Oct 25 17:57 outputs/lora-r16/adapter_model.safetensors
-rw------- 1 root root 264308896 Oct 25 18:22 outputs/lora-r32/adapter_model.safetensors
-rw------- 1 root root  66126768 Oct 25 17:17 outputs/lora-r8/adapter_model.safetensors


## 7 · Sanity check: adapter on vs. off
Same weights, with the adapter switched on and then off, for the last rank in `RANKS`. Check the `total` field: both outputs should keep
the decimal point now.

In [33]:
CHECK = f"outputs/lora-r{RANKS[-1]}"
!python src/infer_check.py --adapter {CHECK} --n 3 --compare

Loading weights: 100% 398/398 [00:30<00:00, 13.08it/s]

=== receipt 100 ===
gold : {"company": "SYARIKAT PERNIAGAAN GIN KEE", "date": "02/12/2017", "address": "NO 290, JALAN AIR PANAS. SETAPAK. 53200, KUALA LUMPUR.", "total": "7.42"}
lora : {"company": "SYARIKAT PERNIAGAAN GIN KEE", "date": "02/12/2017", "address": "NO 290, JALAN AIR PANAS, SETAPAK, 53200, KUALA LUMPUR.", "total": "7.42"}
base : {
  "company": "SYARIKAT PERNIAGAAN GIN KEE",
  "date": "02/12/2017",
  "address": "NO 290, JALAN AIR PANAS. SETAPAK, 53200, KUALA LUMPUR.",
  "total": "7.42"
}

=== receipt 500 ===
gold : {"company": "SANYU STATIONERY SHOP", "date": "02/12/2017", "address": "NO. 31G&33G, JALAN SETIA INDAH X ,U13/X 40170 SETIA ALAM", "total": "8.70"}
lora : {"company": "SANYU STATIONERY SHOP", "date": "02/12/2017", "address": "NO. 31G&33G, JALAN SETIA INDAH X ,U13/X 40170 SETIA ALAM", "total": "8.70"}
base : {
  "company": "SANYU STATIONERY SHOP",
  "date": "02/12/2017",
  "address": "NO. 31G&33G, JALAN SETIA I

## 8 · Evaluate base vs. LoRA (Transformers, no vLLM)
`hf_eval.py` is the Colab addition. It uses the same prompts, the same metrics and the same result files as `eval.py`, so `report.py` works unchanged.
It runs without constrained decoding: that's what the vLLM section adds.
The base model is evaluated once (with the first adapter); every adapter is then evaluated on the same receipts.

Two scores to read side by side: `address_exact` counts punctuation differences as errors, while `address_loose` compares
letters and digits only, because SROIE's address labels are inconsistent about punctuation. `all_loose` is the share of receipts
with all four fields right when the address is scored that way.

In [34]:
for i, r in enumerate(RANKS):
    skip = "" if i == 0 else "--skip-base"     # the base model only needs evaluating once
    !python src/hf_eval.py --adapter outputs/lora-r{r} --n {EVAL_N} --batch-size 8 {skip}

Loading weights: 100% 398/398 [00:30<00:00, 13.10it/s]
  base-hf: 8/100
  base-hf: 16/100
  base-hf: 24/100
  base-hf: 32/100
  base-hf: 40/100
  base-hf: 48/100
  base-hf: 56/100
  base-hf: 64/100
  base-hf: 72/100
  base-hf: 80/100
  base-hf: 88/100
  base-hf: 96/100
  base-hf: 100/100

== base-hf ==
strict_json 1.000   json_valid 1.000   schema_valid 1.000
company  exact 0.790   sim 0.889
date     exact 0.950   sim 0.979
address  exact 0.520   sim 0.934
total    exact 1.000   sim 1.000
address  ignoring punctuation 0.840   all fields (address ignoring punctuation) 0.680
macro    exact 0.815   sim 0.951   all-fields-correct 0.410

  lora-r16-hf: 8/100
  lora-r16-hf: 16/100
  lora-r16-hf: 24/100
  lora-r16-hf: 32/100
  lora-r16-hf: 40/100
  lora-r16-hf: 48/100
  lora-r16-hf: 56/100
  lora-r16-hf: 64/100
  lora-r16-hf: 72/100
  lora-r16-hf: 80/100
  lora-r16-hf: 88/100
  lora-r16-hf: 96/100
  lora-r16-hf: 100/100

== lora-r16-hf ==
strict_json 1.000   json_valid 1.000   schema_valid 1.

In [35]:
!python src/report.py

| run | strict_json | json_valid | schema_valid | company_exact | date_exact | address_exact | total_exact | address_loose | macro_exact | macro_sim | all_exact | all_loose | req_per_s | latency_p50_s |
|---|---|---|---|---|---|---|---|---|---|---|---|---|---|---|
| base-hf | 1.0 | 1.0 | 1.0 | 0.79 | 0.95 | 0.52 | 1.0 | 0.84 | 0.815 | 0.9505 | 0.41 | 0.68 | 0.59 |  |
| lora-r16-hf | 1.0 | 1.0 | 1.0 | 0.96 | 0.99 | 0.74 | 1.0 | 0.95 | 0.9225 | 0.9952 | 0.71 | 0.9 | 0.5 |  |
| lora-r32-hf | 1.0 | 1.0 | 1.0 | 0.95 | 0.99 | 0.74 | 1.0 | 0.96 | 0.92 | 0.9929 | 0.7 | 0.9 | 0.5 |  |
| lora-r8-hf | 1.0 | 1.0 | 1.0 | 0.96 | 0.99 | 0.72 | 1.0 | 0.93 | 0.9175 | 0.993 | 0.7 | 0.89 | 0.5 |  |


In [36]:
for r in RANKS:
    print(f"\n===== worst receipts for rank {r} =====")
    !python src/report.py --errors lora-r{r}-hf -k 3


===== worst receipts for rank 16 =====

--- 253  json_valid=1.0
  company  pred='TOKYO KITCHEN SDN BHD'
           gold='TOKYO KITCHEN (CITTA MALL) TOKYO KITCHEN SDN BHD'
  address  pred='G-26, GRD FLOOR,CITTA MALL, NO 1, JALAN PJU, 1A/4B, JLN PJU 1,ARA DAMANSARA, 47301 PETALING JAYA. SELANGOR.'
           gold='G-26, GRD FLOOR,CITTA MALL, NO 1, JALAN PJU, 1A/4B, JLN PJU 1,ARA DAMANSARA, 47301 PETALING JAYA, SELANGOR.'

--- 042  json_valid=1.0
  company  pred='GEOVENTURE TASTE OF THE WORLD SDN BHD'
           gold='TASTE OF THE WORLD SDN BHD'

--- 192  json_valid=1.0
  date     pred='28-03-18'
           gold='28/03/18'
  address  pred='LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG , 43300 SERI KEMBANGAN , SELANGOR'
           gold='LOT 1851-A & 1851-B, JALAN KPB 6, KAWASAN PERINDUSTRIAN BALAKONG, 43300 SERI KEMBANGAN, SELANGOR'

===== worst receipts for rank 32 =====

--- 519  json_valid=1.0
  company  pred='BHPETROL PERMAS JAYA 2'
           gold='ESJAY FUEL ENTER

## 9 · Save the adapters (the Colab runtime is wiped when it disconnects)
Run either cell. Checkpoint folders (with optimiser state) are skipped.

In [37]:
# Option A: copy to Google Drive
import shutil
from google.colab import drive
drive.mount("/content/drive")
dst = "/content/drive/MyDrive/qwen-receipt-lora"
shutil.copytree("outputs", f"{dst}/outputs", ignore=shutil.ignore_patterns("checkpoint-*"), dirs_exist_ok=True)
shutil.copytree("results", f"{dst}/results", dirs_exist_ok=True)
print("saved to", dst)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
saved to /content/drive/MyDrive/qwen-receipt-lora


In [29]:
# Option B: download one zip with all adapters (+ results)
import shutil
from google.colab import files
shutil.copytree("outputs", "/tmp/export/outputs", ignore=shutil.ignore_patterns("checkpoint-*"), dirs_exist_ok=True)
shutil.copytree("results", "/tmp/export/results", dirs_exist_ok=True)
files.download(shutil.make_archive("lora-adapters", "zip", "/tmp/export"))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 10 · Optional: serve with vLLM and run `eval.py` / `bench.py`
Serves the base model plus **every adapter in `outputs/`** (for example `lora-r8` and `lora-r16` as `sroie-r8` and `sroie-r16`) from one server,
then evaluates and benchmarks each one. Section 6 trains every rank in `RANKS`, so all of them are compared here.
vLLM pins its own PyTorch, so it goes into a separate virtual environment (`/content/.venv-serve`), the same two-environment split as the project.
The training environment stays untouched.

If the version check below shows `GPU visible: False`, or the server log shows a CUDA, driver or `libcudart` error, the vLLM wheel doesn't match Colab's CUDA.
Set `VLLM_SPEC` to an older release built for Colab's CUDA 12.x (for example `"vllm==0.19.0"`) and rerun this section.

In [39]:
VLLM_SPEC = "vllm"
!pip install -q uv
!uv venv -q /content/.venv-serve
!uv pip install -q --python /content/.venv-serve/bin/python "{VLLM_SPEC}" openai rapidfuzz
!/content/.venv-serve/bin/python -c "import torch, vllm; print('vllm', vllm.__version__, '| torch', torch.__version__, 'CUDA', torch.version.cuda, '| GPU visible:', torch.cuda.is_available())"

vllm 0.32.0 | torch 2.13.0+cu130 CUDA 13.0 | GPU visible: True


In [40]:
# Serve the base model + EVERY trained adapter found in outputs/ (lora-r8, lora-r16, ...) and wait until it answers.
import glob, json, os, time, urllib.request
adapter_dirs = sorted(
    (d for d in glob.glob("outputs/lora-*") if os.path.exists(f"{d}/adapter_config.json")),
    key=lambda d: json.load(open(f"{d}/adapter_config.json"))["r"])
assert adapter_dirs, "No trained adapters in outputs/ - train first (section 6) or copy them back from Drive."
ADAPTERS = {"sroie-" + os.path.basename(d).removeprefix("lora-"): d for d in adapter_dirs}  # sroie-r8 -> outputs/lora-r8
max_r = max(json.load(open(f"{d}/adapter_config.json"))["r"] for d in adapter_dirs)
extra = [] if BF16_NATIVE else ["--dtype", "half"]
if max_r > 32:   # serve.sh sets --max-lora-rank 32; a flag given later overrides it
    extra += ["--max-lora-rank", str(next(x for x in (64, 128, 256, 320, 512) if x >= max_r))]
print("adapters to serve:", ADAPTERS)

env = dict(os.environ,
           PATH="/content/.venv-serve/bin:" + os.environ["PATH"],
           ADAPTERS=" ".join(f"{name}={d}" for name, d in ADAPTERS.items()),
           EXTRA_ARGS=" ".join(extra))
log = open("vllm.log", "w")
server = subprocess.Popen(["bash", "scripts/serve.sh"], env=env, stdout=log, stderr=subprocess.STDOUT)
for _ in range(180):
    try:
        data = json.load(urllib.request.urlopen("http://localhost:8000/v1/models"))
        print("server up:", [(m["id"], m.get("parent")) for m in data["data"]])
        break
    except Exception:
        if server.poll() is not None:
            print(open("vllm.log").read()[-4000:])
            raise RuntimeError("vLLM exited - see the log above")
        time.sleep(5)
else:
    print("still starting - check with: !tail -n 40 vllm.log")

adapters to serve: {'sroie-r8': 'outputs/lora-r8', 'sroie-r16': 'outputs/lora-r16', 'sroie-r32': 'outputs/lora-r32'}
server up: [('Qwen/Qwen3-4B-Instruct-2507', None), ('sroie-r8', 'Qwen/Qwen3-4B-Instruct-2507'), ('sroie-r16', 'Qwen/Qwen3-4B-Instruct-2507'), ('sroie-r32', 'Qwen/Qwen3-4B-Instruct-2507')]


In [41]:
# Accuracy for the base model and every adapter, each with free generation and with JSON-schema decoding.
PY = "/content/.venv-serve/bin/python"
BASE = "Qwen/Qwen3-4B-Instruct-2507"
for model, tag in [(BASE, "base-vllm")] + [(name, f"{name}-vllm") for name in ADAPTERS]:
    !{PY} src/eval.py --model {model} --tag {tag} --max-samples {EVAL_N}
    !{PY} src/eval.py --model {model} --tag {tag}-json --structured --max-samples {EVAL_N}


== base-vllm  (Qwen/Qwen3-4B-Instruct-2507, structured=False) ==
strict_json 1.000   json_valid 1.000   schema_valid 1.000
company  exact 0.790   sim 0.889
date     exact 0.950   sim 0.979
address  exact 0.510   sim 0.934
total    exact 1.000   sim 1.000
address  ignoring punctuation 0.840   all fields (address ignoring punctuation) 0.680
macro    exact 0.812   sim 0.950   all-fields-correct 0.400
1.49 req/s, p50 10.394s, p95 12.698s

== base-vllm-json  (Qwen/Qwen3-4B-Instruct-2507, structured=True) ==
strict_json 1.000   json_valid 1.000   schema_valid 1.000
company  exact 0.790   sim 0.889
date     exact 0.950   sim 0.979
address  exact 0.500   sim 0.934
total    exact 1.000   sim 1.000
address  ignoring punctuation 0.840   all fields (address ignoring punctuation) 0.680
macro    exact 0.810   sim 0.950   all-fields-correct 0.400
1.44 req/s, p50 10.762s, p95 13.513s

== sroie-r8-vllm  (sroie-r8, structured=False) ==
strict_json 1.000   json_valid 1.000   schema_valid 1.000
company  

In [42]:
# Serving speed: each adapter alone, then all adapters mixed in the same batches.
for name in ADAPTERS:
    !{PY} src/bench.py --models {name} --concurrency 1,8 --n 32
ALL = ",".join(ADAPTERS)
if len(ADAPTERS) > 1:
    !{PY} src/bench.py --models {ALL} --concurrency 8 --n 32
!python src/report.py

{"models": "sroie-r8", "concurrency": 1, "n": 32, "req_per_s": 0.12, "out_tok_per_s": 9.8, "ttft_p50_ms": 523.3, "tpot_p50_ms": 96.08, "latency_p50_s": 8.638}
{"models": "sroie-r8", "concurrency": 8, "n": 32, "req_per_s": 0.75, "out_tok_per_s": 64.2, "ttft_p50_ms": 361.6, "tpot_p50_ms": 112.86, "latency_p50_s": 9.813}
{"models": "sroie-r16", "concurrency": 1, "n": 32, "req_per_s": 0.12, "out_tok_per_s": 9.8, "ttft_p50_ms": 527.1, "tpot_p50_ms": 96.1, "latency_p50_s": 8.605}
{"models": "sroie-r16", "concurrency": 8, "n": 32, "req_per_s": 0.76, "out_tok_per_s": 64.4, "ttft_p50_ms": 359.1, "tpot_p50_ms": 112.59, "latency_p50_s": 9.778}
{"models": "sroie-r32", "concurrency": 1, "n": 32, "req_per_s": 0.11, "out_tok_per_s": 9.8, "ttft_p50_ms": 530.7, "tpot_p50_ms": 96.22, "latency_p50_s": 8.603}
{"models": "sroie-r32", "concurrency": 8, "n": 32, "req_per_s": 0.75, "out_tok_per_s": 64.2, "ttft_p50_ms": 360.3, "tpot_p50_ms": 112.84, "latency_p50_s": 9.808}
{"models": "sroie-r8+sroie-r16+sroie-

In [43]:
server.terminate(); server.wait(); print('vLLM stopped')

vLLM stopped


## Troubleshooting
| Symptom | Fix |
|---|---|
| `CUDA out of memory` in training | Lower `--batch-size` to 1 and raise `--grad-accum` to 16 in section 5. Make sure no vLLM server is still running. |
| `loss` is `nan` on a T4 | fp16 overflow. Try `--lr 1e-4`, or switch to an L4/A100 runtime (bf16). |
| LoRA output identical to base in section 7 | The quick run is too short to show much. Run with `QUICK = False`. |
| Comparing with results from before the prompt fix | Not comparable: the old prompt said "digits only" for `total`. Retrain the adapters and re-evaluate. |
| Session disconnects during training | Train fewer ranks per session (`RANKS = [16]`) and save each to Drive (section 9). Copy `outputs/` back before section 7. |
| Numbers differ a little between `*-hf` and `*-vllm` rows | Expected: different kernels and fp16 rounding. Compare within one engine. |
| vLLM: adapter rejected for rank | Section 10 raises `--max-lora-rank` automatically when an adapter's rank is above 32. Check the rank in `adapter_config.json`. |
| `ImportError: Found an incompatible version of torchao` | Run `!pip uninstall -y torchao` (section 3 does this) and rerun the cell. No restart needed. |
| Runtime disconnected | Rerun sections 1–5 (files, install, data). If you saved to Drive, copy `outputs/` back instead of retraining. |